# NLP Preprocessing Engine Task-1

**What is the difference between "Love" and "love" in NLP?**

---


It represents as two different tokens while it represents the same semantic meaning. So, Normalization should be applied before training to avoid redundancy.

**What happens if stopwords are not removed?**

---


In terms of model training, the performance nosedives and it focuses on irrelavent words by increased vocabulary and it hurts the model while performing tasks like classification and search.

**Provide two real-world scenarios where removing stopwords can be harmful.**

---


Sentiment Analysis
Example: "I am not happy" → removing "not" changes meaning completely
Chatbots
Example: "What is the time?" → removing "what" breaks intent

**Explain the difference between stemming and lemmatization.**

---


Stemming just cuts words blindly to make them shorter. Sometimes it works, sometimes it gives weird words.  
Example: "running" → "runn"

---


Lemmatization is smarter. It actually understands the word and gives the correct base/root form.  
Example: "running" → "run"

In [11]:
import re
from collections import Counter
#Task 2: Preprocessing Function
def preprocess_text(text):
    # handle edge cases first
    if not isinstance(text,str) or text.strip()=="":
        return [],""
    # lowercase so everything is consistent
    text=text.lower()
    # remove urls
    text=re.sub(r'http\S+|www\S+','',text)
    # remove emails if any
    text=re.sub(r'\S+@\S+','',text)
    # remove numbers
    text=re.sub(r'\d+','',text)
    # fix stretched words like soooo → so
    text=re.sub(r'(.)\1{2,}',r'\1',text)
    # remove emojis/special chars keep only letters
    text=re.sub(r'[^a-z\s]','',text)
    # remove extra spaces
    text=re.sub(r'\s+',' ',text).strip()
    # split into tokens
    tokens=text.split()
    # remove small useless words but keep 'no','not'
    tokens=[w for w in tokens if len(w)>2 or w in ['no','not']]
    # join back
    clean_sentence=" ".join(tokens)
    return tokens,clean_sentence

In [12]:
#Task 3: Stress Testing
test_sentences=[
"Get 100% FREE access now!!!",
"I absolutely looooved this product 😍😍",
"Worst service ever... 0/10",
"Call me at 9876543210",
"This is THE best course!!!",
"Visit https://openai.com now!",
"Nooooo this is baaad!!!",
"OK OK OK I got it",
"Win $$$ now!!! Limited offer!!!",
"I am not happy with this"
]

print("----- Stress Test -----")
results=[]
for text in test_sentences:
    tokens,cleaned=preprocess_text(text)
    results.append((text,tokens,cleaned))
    print("\nOriginal:",text)
    print("Tokens:",tokens)
    print("Cleaned:",cleaned)

----- Stress Test -----

Original: Get 100% FREE access now!!!
Tokens: ['get', 'free', 'access', 'now']
Cleaned: get free access now

Original: I absolutely looooved this product 😍😍
Tokens: ['absolutely', 'loved', 'this', 'product']
Cleaned: absolutely loved this product

Original: Worst service ever... 0/10
Tokens: ['worst', 'service', 'ever']
Cleaned: worst service ever

Original: Call me at 9876543210
Tokens: ['call']
Cleaned: call

Original: This is THE best course!!!
Tokens: ['this', 'the', 'best', 'course']
Cleaned: this the best course

Original: Visit https://openai.com now!
Tokens: ['visit', 'now']
Cleaned: visit now

Original: Nooooo this is baaad!!!
Tokens: ['no', 'this', 'bad']
Cleaned: no this bad

Original: OK OK OK I got it
Tokens: ['got']
Cleaned: got

Original: Win $$$ now!!! Limited offer!!!
Tokens: ['win', 'now', 'limited', 'offer']
Cleaned: win now limited offer

Original: I am not happy with this
Tokens: ['not', 'happy', 'with', 'this']
Cleaned: not happy with this

In [14]:
#Task 4: Token Analytics
def token_analytics(tokens):
    # basic stats on tokens
    if not tokens:
        return 0,0,0
    total=len(tokens)
    unique=len(set(tokens))
    avg_len=sum(len(t) for t in tokens)/total
    return total,unique,round(avg_len,2)

print("\n----- Token Analytics -----")
analytics=[]
for text,tokens,cleaned in results:
    total,unique,avg=token_analytics(tokens)
    analytics.append((text,total,unique,avg))
    print("\nText:",text)
    print("Total:",total,"Unique:",unique,"Avg Length:",avg)

# quick observations
most_noise=max(analytics,key=lambda x:len(x[0]))
most_meaningful=max(analytics,key=lambda x:x[1])

print("\nMost noisy sentence:",most_noise[0])
print("Most meaningful tokens retained:",most_meaningful[0])


----- Token Analytics -----

Text: Get 100% FREE access now!!!
Total: 4 Unique: 4 Avg Length: 4.0

Text: I absolutely looooved this product 😍😍
Total: 4 Unique: 4 Avg Length: 6.5

Text: Worst service ever... 0/10
Total: 3 Unique: 3 Avg Length: 5.33

Text: Call me at 9876543210
Total: 1 Unique: 1 Avg Length: 4.0

Text: This is THE best course!!!
Total: 4 Unique: 4 Avg Length: 4.25

Text: Visit https://openai.com now!
Total: 2 Unique: 2 Avg Length: 4.0

Text: Nooooo this is baaad!!!
Total: 3 Unique: 3 Avg Length: 3.0

Text: OK OK OK I got it
Total: 1 Unique: 1 Avg Length: 3.0

Text: Win $$$ now!!! Limited offer!!!
Total: 4 Unique: 4 Avg Length: 4.5

Text: I am not happy with this
Total: 4 Unique: 4 Avg Length: 4.0

Most noisy sentence: I absolutely looooved this product 😍😍
Most meaningful tokens retained: Get 100% FREE access now!!!


In [16]:
all_tokens=[]
for _,tokens,_ in results:
    all_tokens.extend(tokens)
counter=Counter(all_tokens)
print("\n----- Frequency Analysis -----")
print("Top 10 frequent:",counter.most_common(10))
print("Least 5 frequent:",counter.most_common()[-5:])


----- Frequency Analysis -----
Top 10 frequent: [('this', 4), ('now', 3), ('get', 1), ('free', 1), ('access', 1), ('absolutely', 1), ('loved', 1), ('product', 1), ('worst', 1), ('service', 1)]
Least 5 frequent: [('limited', 1), ('offer', 1), ('not', 1), ('happy', 1), ('with', 1)]


In [29]:
#Task 6: Build Full Pipeline
def full_pipeline(text_list):
    all_tokens=[]
    clean_sentences=[]
    for text in text_list:
        tokens, cleaned=preprocess_text(text)
        all_tokens.extend(tokens)
        clean_sentences.append(cleaned)
    return {
        "tokens":all_tokens,
        "clean_sentences":clean_sentences
    }
result=full_pipeline(test_sentences)
print("All Tokens:")
print(result["tokens"])
print("\nClean Sentences:")
for sentence in result["clean_sentences"]:
    print(sentence)

All Tokens:
['get', 'free', 'access', 'now', 'absolutely', 'loved', 'this', 'product', 'worst', 'service', 'ever', 'call', 'this', 'the', 'best', 'course', 'visit', 'now', 'no', 'this', 'bad', 'got', 'win', 'now', 'limited', 'offer', 'not', 'happy', 'with', 'this']

Clean Sentences:
get free access now
absolutely loved this product
worst service ever
call
this the best course
visit now
no this bad
got
win now limited offer
not happy with this


In [20]:
#Task 7: Error Handling
edge_cases=["","😂😂😂😂","123456789"]

print("\n----- Edge Case Testing -----")
for case in edge_cases:
    tokens,cleaned=preprocess_text(case)
    print("\nInput:",case)
    print("Tokens:",tokens)
    print("Cleaned:",cleaned)


----- Edge Case Testing -----

Input: 
Tokens: []
Cleaned: 

Input: 😂😂😂😂
Tokens: []
Cleaned: 

Input: 123456789
Tokens: []
Cleaned: 
